# Ask the knowledge assistant
Retrieves the top chunks, optionally restricted to one page through a metadata filter, and answers with citations. Each question is logged to `qa_log`.

In [ ]:
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("schema", "")
dbutils.widgets.text("src_path", "")

import sys

sys.path.insert(0, dbutils.widgets.get("src_path"))
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
target = f"{catalog}.{schema}"

dbutils.widgets.text("llm_endpoint", "")

from rag_kb.service import Assistant, log_interactions

assistant = Assistant(f"{target}.docs_chunks_index", dbutils.widgets.get("llm_endpoint"))

dbutils.widgets.text("question", "How do I remove old files from a Delta table and how long are they kept by default?")
dbutils.widgets.dropdown("chunk_profile", "small", ["small", "large"])
dbutils.widgets.text("doc_path", "")
dbutils.widgets.text("top_k", "5")

In [ ]:
result = assistant.ask(
    dbutils.widgets.get("question"),
    top_k=int(dbutils.widgets.get("top_k")),
    profile=dbutils.widgets.get("chunk_profile"),
    doc_path=dbutils.widgets.get("doc_path") or None,
)
log_interactions(spark, f"{target}.qa_log", [result])

print(result["answer"])
print()
for i, chunk in enumerate(result["chunks"], start=1):
    print(f"[{i}] {chunk['score']:.3f} {chunk['doc_path']} > {chunk['section'] or 'Overview'}")